In [1]:
import pandas as pd
import re, os, joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score


In [2]:
df = pd.read_excel('./data/sinhala_biz_news.xlsx').head(3002)
df['news_content'] = df['news_content'].fillna('')

In [3]:
df.head()

,index,date,headline,news_content,MCPL.N0000,KOTA.N0000,HAPU.N0000,WATA.N0000,AGPL.N0000,BFL.N0000,RWSL.N0000,DIPP.N0000,MGT.N0000,HEXP.N0000,Unnamed: 14,Unnamed: 15,Unnamed: 16,url
0,1,2025-09-04 06:11:18,ඩිජිටල් සේවා සඳහා වැට් බද්ද බලාත්මක කිරීම කල් යයි,2025 අංක 4 දරන එකතු කළ අගය මත බදු (සංශෝධන) පනත...,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,NaN,NaN,NaN,http://biz.adaderana.lk/%e0%b6%a9%e0%b7%92%e0%...
1,2,NaN,පාඩු ලබන රාජ්‍ය ව්‍යවසාය 33ක් වසා දැමීමට කැබින...,මහජන සේවා සැපයීම සහ උපායමාර්ගික ආර්ථික ක්‍රියා...,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,NaN,NaN,NaN,http://biz.adaderana.lk/%e0%b6%b4%e0%b7%8f%e0%...
2,3,2025-09-03 10:53:35,මෙරට නිෂ්පාදිත ඖෂධ ලෝක වෙළඳපොලට,• 100%ක දේශීය සමාගමක් ඖෂධ අපනයනයට එක් වූ මුල් ...,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,NaN,NaN,NaN,http://biz.adaderana.lk/%e0%b6%b8%e0%b7%99%e0%...
3,4,2025-09-03 10:40:46,ශ්‍රී ලංකාවේ ප්‍රථම විදුලි සංදේශ යටිතල පහසුකම්...,EDOTCO ආයතනයට ශ්‍රී ලංකාවේ ප්‍රථම විදුලි සංදේශ...,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral,NaN,NaN,NaN,http://biz.adaderana.lk/%e0%b7%81%e0%b7%8a%e2%...
4,5,2025-09-03 06:46:31,සංචාරක ක්ෂේත්‍රය ප්‍රවර්ධනය කිරීමේ අරමුණින් කා...,සංචාරක ක්ෂේත්‍රය ප්‍රවර්ධනය කිරීමේ අරමුණින් කා...,Neutral,Neutral,Neutral,Neutral,Neutral,Positive,Positive,Neutral,Neutral,Neutral,NaN,NaN,NaN,http://biz.adaderana.lk/%e0%b7%83%e0%b6%82%e0%...


In [4]:
companies = [
    'WATA.N0000','AGPL.N0000','HAPU.N0000','MCPL.N0000',
    'KOTA.N0000','BFL.N0000','RWSL.N0000','DIPP.N0000',
    'MGT.N0000','HEXP.N0000'
]

df[companies] = df[companies].fillna('Neutral')
X = df['news_content']


In [5]:
stop_words = open('./data/stop words.txt', encoding='utf-8').read().splitlines()

def tokenizer(text):
    tokens = re.findall(r'\b\w+\b', text.lower())
    return [t for t in tokens if t not in stop_words]


In [6]:
class_weights = {
    'Neutral': 1,
    'Positive': 2,
    'Negative': 3
}


In [7]:
os.makedirs("saved_models", exist_ok=True)
results = []

for company in companies:
    y = df[company]

    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(tokenizer=tokenizer, max_features=1000)),
        ('svm', SVC(kernel='linear', class_weight=class_weights))
    ])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )

    pipeline.fit(X_train, y_train)

    model_path = f"saved_models/{company}_svm.joblib"
    joblib.dump(pipeline, model_path)

    results.append({
        "Company": company,
        "Train Acc": accuracy_score(y_train, pipeline.predict(X_train)),
        "Test Acc": accuracy_score(y_test, pipeline.predict(X_test))
    })

pd.DataFrame(results)


D:\FYP_Project\venv\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
D:\FYP_Project\venv\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
D:\FYP_Project\venv\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
D:\FYP_Project\venv\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
D:\FYP_Project\venv\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
D:\FYP_Project\venv\Lib\site-packages\sklearn\feature_extraction\text.

,Company,Train Acc,Test Acc
0,WATA.N0000,0.851975,0.799112
1,AGPL.N0000,0.863398,0.801332
2,HAPU.N0000,0.866730,0.792453
3,MCPL.N0000,0.860067,0.793563
4,KOTA.N0000,0.866254,0.782464
5,BFL.N0000,0.847692,0.795782
6,RWSL.N0000,0.853403,0.817980
7,DIPP.N0000,0.858163,0.790233
8,MGT.N0000,0.859591,0.795782
9,HEXP.N0000,0.857211,0.807991
